#**Importación de Librerías**

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

epocas=5

'''
fue necesario utilizar el dataset de fashion_mnist
debido a que en el momento del desarrollo de este
ejercicio del servidor de Toronto en el que se encuentra
cifar10 no se encontraba disponible
'''

'''(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()

X_train=X_train / 255.0
X_test=X_test / 255.0

'''

In [ ]:
(X_train, y_train), (X_test, y_test)=keras.datasets.fashion_mnist.load_data()

X_train=X_train / 255.0
X_test=X_test / 255.0

'''
Las dos lineas a continuación se deben agregar al utilizar
el dataset de fashion_mnist
el objetivo es convertir a 3 canales
para CNN t transfer leraning
'''

X_train=tf.image.grayscale_to_rgb(tf.expand_dims(X_train, -1))
X_test=tf.image.grayscale_to_rgb(tf.expand_dims(X_test, -1))

#Visualización

In [ ]:
plt.imshow(X_train[0])
plt.title(f"Clase: {y_train}")
plt.axis("off")
plt.show()

#Data Augmentation

In [ ]:
data_augmentation=keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

#Visualización de Augmentation

In [ ]:
for i in range(5):
  augmented=data_augmentation(tf.expand_dims(X_train[0],0))
  plt.imshow(augmented[0])
  plt.axis("off")
  plt.show()

#Modelo Base

In [ ]:
def modelo_base():
  model=keras.Sequential([
      keras.Input(shape=(28,28,3)),

      layers.Conv2D(32, (3,3), activation="relu", input_shape=(32,32,3)),
      layers.MaxPooling2D((2,2)),

      layers.Conv2D(64, (3,3), activation="relu"),
      layers.MaxPooling2D((2,2)),

      layers.Flatten(),
      layers.Dense(64, activation="relu"),
      layers.Dense(10, activation="softmax")
  ])
  model.compile(
      optimizer="adam",
      loss="sparse_categorical_crossentropy",
      metrics=["accuracy"]
  )
  return model

model_base=modelo_base()

history_base=model_base.fit(
    X_train, y_train,
    epochs=epocas,
    validation_data=(X_test, y_test)
)

#***Modelo Data Augmentation***

In [ ]:
def modelo_aug():
  model=keras.Sequential([
      data_augmentation,

      layers.Conv2D(32,(3,3), activation='relu'),
      layers.MaxPooling2D(2,2),

      layers.Conv2D(64,(3,3), activation='relu'),
      layers.MaxPooling2D(2,2),

      layers.Flatten(),
      layers.Dense(64, activation='relu'),
      layers.Dense(10, activation='softmax')

  ])
  model.compile(
      optimizer="adam",
      loss="sparse_categorical_crossentropy",
      metrics=["accuracy"]
  )
  return model

model_aug=modelo_aug()

history_aug=model_aug.fit(
    X_train, y_train,
    epochs=epocas,
    validation_data=(X_test, y_test)
)

#Transfer Learning
MobileNetV2

In [ ]:
X_train_resized=tf.image.resize(X_train, (96,96))
X_test_resized=tf.image.resize(X_test, (96,96))

base_model=keras.applications.MobileNetV2(
    input_shape=(96,96,3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable=False

model_tl=keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model_tl.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_tl=model_tl.fit(
    X_train_resized, y_train,
    epochs=epocas,
    validation_data=(X_test_resized, y_test)
)

#Graficas de Comparación

In [ ]:
plt.plot(history_base.history['val_accuracy'], label='Base')
plt.plot(history_aug.history['val_accuracy'], label='Augmentation')
plt.plot(history_tl.history['val_accuracy'], label='Transfer Learning')

plt.titl("Comparación de Modelos")
plt.xlabel("Épocas")
plt.ylabel("Precisión de Validación")
plt.legend()
plt.show()
